# Feature Engineering

## Detección de fraude en establecimientos de hospedaje

Este notebook construye las variables derivadas para el modelo.  
**Regla fundamental**: todos los agregados históricos se calculan exclusivamente
con transacciones **anteriores** a la actual — cero *look-ahead bias*.

### Técnica vectorizada (sin look-ahead)
```python
# Para la transacción N de un cliente, solo se usan las N-1 anteriores:
cumsum_hasta_actual  = df.groupby('client_id')['amount_usd'].cumsum()
suma_antes           = cumsum_hasta_actual - df['amount_usd']  # excluye la fila actual
promedio_antes       = suma_antes / df.groupby('client_id').cumcount().replace(0, float('nan'))
```
Esta formulación es algebraicamente equivalente a `expanding().mean().shift(1)` dentro de
cada grupo, pero es completamente vectorizada (sin lambdas ni apply).


## 1. Importaciones y carga de datos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


In [2]:
df = pd.read_csv('../data/processed/eda_base_dataset.csv')

# Restaurar tipos que CSV serializa como string
df['transaction_datetime'] = pd.to_datetime(
    df['DE7_transmission_datetime'].astype(str).str.zfill(10),
    format='%m%d%H%M%S', errors='coerce'
)
for col in ['is_fraud', 'is_international', 'is_hotel',
            'hotel_international', 'hotel_high_distance']:
    df[col] = df[col].astype(bool)

print(f'Cargado: {df.shape[0]:,} filas × {df.shape[1]} cols')
print(f'Clientes únicos: {df["client_id"].nunique():,}')
print(f'Meses: {sorted(df["transaction_month"].dropna().unique().astype(int))}')


Cargado: 100,003 filas × 71 cols
Clientes únicos: 4,000
Meses: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


## 2. Confirmación de MCCs de hospedaje (PlusTI)

Cruzamos el rango estándar ISO 8583 (3501–3999 + 7011) contra los MCCs
efectivamente presentes en el dataset, y validamos contra los nombres de comercio.

In [3]:
iso_hotel_range = set(range(3501, 4000)) | {7011}
mccs_en_data    = set(df['DE18_merchant_category_code'].unique())
rango_presente  = sorted(iso_hotel_range & mccs_en_data)

print('MCCs del rango ISO 3501-3999 en el dataset PlusTI:')
print(rango_presente if rango_presente else '  → Ninguno del rango 3501-3999')
print(f'MCC 7011 presente: {7011 in mccs_en_data}')

# Validación cruzada por nombre de comercio
keywords = ['HOTEL','HOSTEL','INN','RESORT','MOTEL','LODGE','SUITES',
            'HILTON','MARRIOTT','HYATT','SHERATON','RADISSON','IBIS','NOVOTEL']
mask_nombre = df['DE43_card_acceptor_name_location'].str.upper().str.contains(
    '|'.join(keywords), na=False)

print()
print('MCCs en comercios con nombre de hotel/hospedaje:')
print(df[mask_nombre]['DE18_merchant_category_code'].value_counts())
print()
print('Conclusión PlusTI: ÚNICO MCC de hospedaje en este dataset es 7011.')
print('El rango 3501-3999 (cadenas individuales ISO) no está codificado aquí.')


MCCs del rango ISO 3501-3999 en el dataset PlusTI:
[np.int64(7011)]
MCC 7011 presente: True

MCCs en comercios con nombre de hotel/hospedaje:
DE18_merchant_category_code
7011    9064
Name: count, dtype: int64

Conclusión PlusTI: ÚNICO MCC de hospedaje en este dataset es 7011.
El rango 3501-3999 (cadenas individuales ISO) no está codificado aquí.


In [4]:
# Corrección: is_hotel = solo MCC 7011 (confirmado con datos PlusTI)
HOTEL_MCCS = [7011]
df['is_hotel'] = df['DE18_merchant_category_code'].isin(HOTEL_MCCS)

print(f'Transacciones hospedaje (MCC 7011): {df["is_hotel"].sum():,}')
print(f'Tasa de fraude en hospedaje:        {df[df["is_hotel"]]["is_fraud"].mean()*100:.2f}%')


Transacciones hospedaje (MCC 7011): 9,064
Tasa de fraude en hospedaje:        4.58%


## 3. Ordenamiento temporal

**Paso crítico**: ordenar por `client_id` + `transaction_datetime` garantiza
que las operaciones acumuladas siempre avancen en el tiempo correcto por cliente.

In [5]:
df = df.sort_values(['client_id', 'transaction_datetime']).reset_index(drop=True)

orden_ok = df.groupby('client_id')['transaction_datetime'].apply(
    lambda x: (x.diff().dropna() >= pd.Timedelta(0)).all()
).all()
print(f'Orden temporal garantizado por cliente: {orden_ok}')


Orden temporal garantizado por cliente: True


## 4. Features históricas por cliente

**Estrategia vectorizada**: `cumsum_hasta_actual - valor_actual` = suma de las N-1 anteriores.  
Para la transacción #1 de cada cliente: count_before=0 → divisor NaN → promedio NaN ✅

In [6]:
g = df.groupby('client_id')

# ── Conteo de transacciones anteriores ───────────────────────────────────
df['client_txn_count_before'] = g.cumcount()   # 0 en la primera txn

# ── Promedio histórico de monto ───────────────────────────────────────────
# cumsum incluye la fila actual → restamos el valor actual para excluirla
cumsum_amount = g['amount_usd'].cumsum()
df['client_avg_amount_before'] = (
    (cumsum_amount - df['amount_usd'])
    / df['client_txn_count_before'].replace(0, np.nan)
)

# ── Std y máximo histórico (expanding per-group via transform) ─────────────
# expanding().shift(1) dentro del transform opera sobre la serie del grupo,
# por lo que el shift es relativo al grupo, no global → correcto
df['client_std_amount_before'] = g['amount_usd'].transform(
    lambda x: x.expanding().std().shift(1)
)
df['client_max_amount_before'] = g['amount_usd'].transform(
    lambda x: x.expanding().max().shift(1)
)

# ── Ratios respecto al historial ─────────────────────────────────────────
df['amount_vs_hist_avg']   = df['amount_usd'] / df['client_avg_amount_before'].replace(0, np.nan)
df['amount_over_hist_std'] = (
    (df['amount_usd'] - df['client_avg_amount_before'])
    / df['client_std_amount_before'].replace(0, np.nan)
)  # z-score respecto al historial personal

print('Promedio histórico de monto por is_fraud:')
print(df.groupby('is_fraud')['client_avg_amount_before'].mean().round(2))


Promedio histórico de monto por is_fraud:
is_fraud
False    422.38
True     471.55
Name: client_avg_amount_before, dtype: float64


In [7]:
# ── Tasa histórica de transacciones internacionales ──────────────────────
intl_int         = df['is_international'].astype(int)
cumsum_intl      = intl_int.groupby(df['client_id']).cumsum()
df['client_intl_rate_before'] = (
    (cumsum_intl - intl_int)
    / df['client_txn_count_before'].replace(0, np.nan)
)

print('Tasa histórica internacional - fraude vs legítimo:')
print(df.groupby('is_fraud')['client_intl_rate_before'].mean().round(4))


Tasa histórica internacional - fraude vs legítimo:
is_fraud
False    0.2322
True     0.2614
Name: client_intl_rate_before, dtype: float64


## 5. Features históricas por canal

Promedio y conteo en el **mismo canal** (POS / ECOM / ATM / MOTO),
calculado solo con las transacciones anteriores del cliente en ese canal.

In [8]:
gc = df.groupby(['client_id', 'channel'])

# Conteo anterior en el mismo canal (0-indexed → count before current)
df['client_channel_txn_count_before'] = gc.cumcount()

# Promedio histórico de monto en el mismo canal
cumsum_ch = gc['amount_usd'].cumsum()
df['client_channel_avg_amount_before'] = (
    (cumsum_ch - df['amount_usd'])
    / df['client_channel_txn_count_before'].replace(0, np.nan)
)

# Ratio monto actual vs promedio del canal
df['amount_vs_channel_avg'] = (
    df['amount_usd'] / df['client_channel_avg_amount_before'].replace(0, np.nan)
)

print('Promedio histórico por canal — fraude vs legítimo:')
print(
    df.groupby(['channel', 'is_fraud'])['client_channel_avg_amount_before']
    .mean().round(2).unstack()
)


Promedio histórico por canal — fraude vs legítimo:
is_fraud   False    True 
channel                  
ATM       206.31   284.20
ECOM      551.64   583.73
MOTO      771.27  1035.06
POS       284.29   335.01


## 6. Features históricas de hospedaje (MCC 7011)

Historial del cliente específicamente en comercios de hospedaje.  
Un cliente sin historial previo en hoteles que aparece en uno es señal de alerta.

In [9]:
gh = df.groupby(['client_id', 'is_hotel'])

# Conteo de transacciones anteriores en hospedaje
df['client_hotel_txn_count_before'] = gh.cumcount()
# Para transacciones no-hotel, el conteo es irrelevante → 0
df.loc[~df['is_hotel'], 'client_hotel_txn_count_before'] = 0

# Promedio histórico de monto en hospedaje
cumsum_hotel = gh['amount_usd'].cumsum()
sum_hotel_before = cumsum_hotel - df['amount_usd']
df['client_hotel_avg_amount_before'] = np.where(
    df['is_hotel'] & (df['client_hotel_txn_count_before'] > 0),
    sum_hotel_before / df['client_hotel_txn_count_before'],
    np.nan
)

# Tasa histórica de hotel internacional
intl_int2         = df['is_international'].astype(int)
cumsum_h_intl     = intl_int2.groupby([df['client_id'], df['is_hotel']]).cumsum()
sum_h_intl_before = cumsum_h_intl - intl_int2
df['client_hotel_intl_rate_before'] = np.where(
    df['is_hotel'] & (df['client_hotel_txn_count_before'] > 0),
    sum_h_intl_before / df['client_hotel_txn_count_before'],
    np.nan
)

# Flag: primera vez en hospedaje para este cliente
df['client_hotel_first_time'] = (
    df['is_hotel'] & (df['client_hotel_txn_count_before'] == 0)
).astype(int)

hotel_mask = df['is_hotel']
print('Historial previo en hospedaje — fraude vs legítimo:')
print(df[hotel_mask].groupby('is_fraud')['client_hotel_txn_count_before'].describe().round(2))
print()
print('Primera vez en hotel (sin historial):')
print(df[hotel_mask].groupby('is_fraud')['client_hotel_first_time'].mean().round(4))


Historial previo en hospedaje — fraude vs legítimo:
           count  mean   std  min  25%  50%  75%  max
is_fraud                                             
False     8649.0  1.11  1.23  0.0  0.0  1.0  2.0  9.0
True       415.0  1.34  1.39  0.0  0.0  1.0  2.0  9.0

Primera vez en hotel (sin historial):
is_fraud
False    0.3995
True     0.3494
Name: client_hotel_first_time, dtype: float64


## 7. Features temporales

Tiempo desde la última transacción del cliente.  
Transacciones muy seguidas pueden indicar fraude (*card testing*).

In [10]:
# shift(1) en SeriesGroupBy opera PER-GROUP → correcto
df['prev_txn_datetime'] = g['transaction_datetime'].shift(1)
df['hours_since_last_txn'] = (
    (df['transaction_datetime'] - df['prev_txn_datetime'])
    .dt.total_seconds() / 3600
)
df['hours_since_last_txn'] = df['hours_since_last_txn'].fillna(-1)  # -1 = primera txn
df.drop(columns=['prev_txn_datetime'], inplace=True)

# Flag de sucesión rápida (<1h desde la última transacción)
df['is_rapid_succession'] = (
    (df['hours_since_last_txn'] >= 0) & (df['hours_since_last_txn'] < 1)
).astype(int)

mask_not_first = df['hours_since_last_txn'] >= 0
print('Horas desde última transacción — fraude vs legítimo:')
print(df[mask_not_first].groupby('is_fraud')['hours_since_last_txn'].describe().round(2))
print()
print('Tasa de sucesión rápida (<1h):')
print(df.groupby('is_fraud')['is_rapid_succession'].mean().round(4))


Horas desde última transacción — fraude vs legítimo:
            count    mean     std  min    25%     50%     75%      max
is_fraud                                                              
False     91156.0  170.74  169.63  0.0  48.87  119.28  237.42  2109.51
True       4847.0   64.06  131.76  0.0   0.04    2.09   69.16  1195.26

Tasa de sucesión rápida (<1h):
is_fraud
False    0.0062
True     0.4119
Name: is_rapid_succession, dtype: float64


## 8. Verificación de integridad temporal (anti look-ahead)

Confirmamos que ninguna feature histórica incluye información de la transacción actual.

In [11]:
# Test 1: en la primera transacción de cada cliente → historial debe ser NaN
primeras = df[df['client_txn_count_before'] == 0]
hist_cols = [
    'client_avg_amount_before', 'client_std_amount_before',
    'client_max_amount_before', 'client_intl_rate_before',
    'client_channel_avg_amount_before',
]
print('Primera transacción de cada cliente — todo historial debe ser NaN:')
for col in hist_cols:
    n_nan = primeras[col].isna().sum()
    total = len(primeras)
    ok = n_nan == total
    print(f'  {col:<42} {"✅" if ok else f"❌ {total-n_nan} no-NaN"}')


Primera transacción de cada cliente — todo historial debe ser NaN:
  client_avg_amount_before                   ✅
  client_std_amount_before                   ✅
  client_max_amount_before                   ✅
  client_intl_rate_before                    ✅
  client_channel_avg_amount_before           ✅


In [12]:
# Test 2: en la segunda transacción, client_avg_amount_before == monto de la primera
segundas = df[df['client_txn_count_before'] == 1].copy()
monto_primera = (
    df[df['client_txn_count_before'] == 0]
    .set_index('client_id')['amount_usd']
)
segundas['expected'] = segundas['client_id'].map(monto_primera)
pct_ok = np.isclose(
    segundas['client_avg_amount_before'],
    segundas['expected'], atol=0.01
).mean() * 100

print(f'Test look-ahead — txn #2: avg_before debe == monto de txn #1')
status = '✅' if pct_ok > 99 else '❌'
print(f'  Coincidencia: {pct_ok:.2f}% {status}')


Test look-ahead — txn #2: avg_before debe == monto de txn #1
  Coincidencia: 100.00% ✅


## 9. Resumen de features generadas

| Feature | Descripción | Garantía temporal |
|---|---|---|
| `client_txn_count_before` | Nº txns previas del cliente | ✅ cumcount() |
| `client_avg_amount_before` | Monto promedio histórico | ✅ cumsum − actual |
| `client_std_amount_before` | Std histórica de montos | ✅ expanding+shift (por grupo) |
| `client_max_amount_before` | Máximo histórico de monto | ✅ expanding+shift (por grupo) |
| `amount_vs_hist_avg` | Ratio monto actual / promedio histórico | ✅ derivada |
| `amount_over_hist_std` | Z-score vs historial del cliente | ✅ derivada |
| `client_intl_rate_before` | Tasa histórica de txns internacionales | ✅ cumsum − actual |
| `client_channel_txn_count_before` | Nº txns previas en el mismo canal | ✅ cumcount() |
| `client_channel_avg_amount_before` | Promedio histórico en el mismo canal | ✅ cumsum − actual |
| `amount_vs_channel_avg` | Ratio vs promedio del canal | ✅ derivada |
| `client_hotel_txn_count_before` | Nº txns previas en hospedaje (7011) | ✅ cumcount() |
| `client_hotel_avg_amount_before` | Promedio histórico en hospedaje | ✅ cumsum − actual |
| `client_hotel_intl_rate_before` | Tasa histórica hotel internacional | ✅ cumsum − actual |
| `client_hotel_first_time` | Sin historial previo en hotel | ✅ derivada |
| `hours_since_last_txn` | Horas desde la txn anterior | ✅ shift(1) por grupo |
| `is_rapid_succession` | Flag: < 1h desde la txn anterior | ✅ derivada |


In [13]:
new_features = [
    'client_txn_count_before', 'client_avg_amount_before',
    'client_std_amount_before', 'client_max_amount_before',
    'amount_vs_hist_avg', 'amount_over_hist_std',
    'client_intl_rate_before',
    'client_channel_txn_count_before', 'client_channel_avg_amount_before',
    'amount_vs_channel_avg',
    'client_hotel_txn_count_before', 'client_hotel_avg_amount_before',
    'client_hotel_intl_rate_before', 'client_hotel_first_time',
    'hours_since_last_txn', 'is_rapid_succession',
]

print(f'Features nuevas: {len(new_features)}')
print(f'Total columnas:  {df.shape[1]}')
print()
print('Correlación con is_fraud (features nuevas, valor absoluto):')
corr = df[new_features + ['is_fraud']].corr(numeric_only=True)['is_fraud'].drop('is_fraud')
print(corr.abs().sort_values(ascending=False).round(4).to_string())


Features nuevas: 16
Total columnas:  87

Correlación con is_fraud (features nuevas, valor absoluto):
is_rapid_succession                 0.5493
hours_since_last_txn                0.1284
client_channel_txn_count_before     0.0939
client_channel_avg_amount_before    0.0680
amount_vs_channel_avg               0.0667
amount_vs_hist_avg                  0.0446
client_avg_amount_before            0.0398
client_max_amount_before            0.0396
client_txn_count_before             0.0366
client_std_amount_before            0.0355
client_intl_rate_before             0.0351
client_hotel_intl_rate_before       0.0316
client_hotel_avg_amount_before      0.0118
client_hotel_first_time             0.0080
client_hotel_txn_count_before       0.0054
amount_over_hist_std                0.0020


## 10. Guardar dataset con features

## 11. Half 1 — Variables de velocidad y desviación

Variables que capturan *cuánto* y *con qué rapidez* transacciona el cliente.
Son las más utilizadas en sistemas de fraude en tiempo real.

### División de trabajo
| Half | Responsable | Enfoque |
|---|---|---|
| Half 1 (esta sección) | Davis | Velocidad + desviación de monto |
| Half 2 (siguiente notebook) | Compañera | Hotel-específicas + comportamiento avanzado |

### 11.1 `time_since_last_txn_min` — Minutos desde la última transacción

**Por qué importa:** el fraude de *card-testing* genera transacciones en ráfaga (segundos/minutos de diferencia). Ya calculamos `hours_since_last_txn`; lo convertimos a minutos para mayor granularidad.

In [14]:
# Conversión directa: horas → minutos
df['time_since_last_txn_min'] = df['hours_since_last_txn'] * 60
# -1 sigue indicando primera transacción del cliente

mask_not_first = df['time_since_last_txn_min'] >= 0
print('Minutos desde última txn — fraude vs legítimo:')
print(df[mask_not_first].groupby('is_fraud')['time_since_last_txn_min']
      .describe().round(2))


Minutos desde última txn — fraude vs legítimo:
            count      mean       std   min      25%      50%       75%  \
is_fraud                                                                  
False     91156.0  10244.24  10177.93  0.30  2932.12  7156.52  14245.31   
True       4847.0   3843.72   7905.33  0.17     2.18   125.52   4149.52   

                max  
is_fraud             
False     126570.52  
True       71715.77  


### 11.2-11.4 `txn_count_last_1h / 24h / 7d` — Velocidad de transacciones

**Por qué importa:** un ladrón que obtiene una tarjeta la usa intensivamente en un período corto antes de que sea bloqueada.  
**Técnica:** para cada transacción contamos cuántas transacciones *previas* del mismo cliente caen dentro de la ventana temporal → garantía anti look-ahead.

In [15]:
from bisect import bisect_left

def sliding_txn_count(group, td):
    """Cuenta txns anteriores del cliente dentro de la ventana td."""
    times = group['transaction_datetime'].values.astype('int64')
    window_ns = int(td.total_seconds() * 1e9)
    n = len(times)
    counts = np.zeros(n, dtype=int)
    for i in range(1, n):
        window_start = times[i] - window_ns
        # bisect sobre times[:i] (solo anteriores) → O(log n)
        left = bisect_left(times, window_start, 0, i)
        counts[i] = i - left  # txns en [window_start, times[i])
    return pd.Series(counts, index=group.index)

df = df.sort_values(['client_id', 'transaction_datetime']).reset_index(drop=True)
g  = df.groupby('client_id', group_keys=False)

print('Calculando txn_count_last_1h ...')
df['txn_count_last_1h'] = g.apply(
    lambda grp: sliding_txn_count(grp, pd.Timedelta(hours=1)))

print('Calculando txn_count_last_24h ...')
df['txn_count_last_24h'] = g.apply(
    lambda grp: sliding_txn_count(grp, pd.Timedelta(hours=24)))

print('Calculando txn_count_last_7d ...')
df['txn_count_last_7d'] = g.apply(
    lambda grp: sliding_txn_count(grp, pd.Timedelta(days=7)))

print('\nConteo medio de txns en ventana — fraude vs legítimo:')
for col in ['txn_count_last_1h','txn_count_last_24h','txn_count_last_7d']:
    means = df.groupby('is_fraud')[col].mean().round(3)
    print(f'  {col:<25} legítimo={means[False]:.3f}  fraude={means[True]:.3f}')


Calculando txn_count_last_1h ...
Calculando txn_count_last_24h ...
Calculando txn_count_last_7d ...

Conteo medio de txns en ventana — fraude vs legítimo:
  txn_count_last_1h         legítimo=0.006  fraude=1.038
  txn_count_last_24h        legítimo=0.139  fraude=1.624
  txn_count_last_7d         legítimo=0.948  fraude=2.436


### 11.5 `amount_zscore_customer` — Z-score del monto respecto al historial del cliente

**Por qué importa:** un monto muy superior a la media histórica del cliente es señal de anomalía.  
**Fórmula:** `(amount_usd − client_avg_before) / client_std_before`  
Esta variable ya fue calculada como `amount_over_hist_std`; aquí la renombramos con el nombre estándar del enunciado.

In [16]:
# Renombre con nombre estándar del enunciado
df['amount_zscore_customer'] = df['amount_over_hist_std']

print('Z-score de monto del cliente — fraude vs legítimo:')
print(df.groupby('is_fraud')['amount_zscore_customer']
      .describe().round(3))


Z-score de monto del cliente — fraude vs legítimo:
            count   mean      std        min    25%    50%    75%        max
is_fraud                                                                    
False     87273.0  0.358  141.229 -38457.417 -0.683 -0.412  0.274  11547.078
True       4730.0  1.611    3.711    -13.815 -0.186  0.964  2.554    128.679


### 11.6 `amount_zscore_channel` — Z-score del monto respecto al canal del cliente

**Por qué importa:** un monto de ECOM anómalo es diferente a un monto de ATM anómalo. Evaluar dentro del canal del cliente filtra el ruido.

In [17]:
# Necesitamos std histórico por canal (análogo a client_channel_avg_amount_before)
gc = df.groupby(['client_id', 'channel'])
df['client_channel_std_before'] = gc['amount_usd'].transform(
    lambda x: x.expanding().std().shift(1)
)

df['amount_zscore_channel'] = (
    (df['amount_usd'] - df['client_channel_avg_amount_before'])
    / df['client_channel_std_before'].replace(0, np.nan)
)

print('Z-score de monto por canal — fraude vs legítimo:')
print(df.groupby('is_fraud')['amount_zscore_channel']
      .describe().round(3))


Z-score de monto por canal — fraude vs legítimo:
            count   mean      std        min    25%    50%    75%        max
is_fraud                                                                    
False     69228.0  1.554  217.206 -28391.044 -0.766 -0.396  0.617  47602.664
True       4199.0  3.064   22.327    -45.995 -0.230  0.921  2.662   1165.666


### 11.7-11.8 `unique_merchants_last_24h` y `unique_countries_last_24h`

**Por qué importa:**
- Muchos comercios distintos en 24h = patrón de *enumeration fraud*.
- Muchos países en 24h = imposible físicamente → *impossible travel*.

In [18]:
def sliding_unique(group, col, td):
    """Cuenta valores únicos de 'col' en txns anteriores dentro de la ventana td."""
    times  = group['transaction_datetime'].values.astype('int64')
    values = group[col].values
    window_ns = int(td.total_seconds() * 1e9)
    n = len(times)
    counts = np.zeros(n, dtype=int)
    for i in range(1, n):
        window_start = times[i] - window_ns
        mask = times[:i] >= window_start  # solo anteriores dentro de la ventana
        counts[i] = len(set(values[:i][mask]))
    return pd.Series(counts, index=group.index)

print('Calculando unique_merchants_last_24h ...')
df['unique_merchants_last_24h'] = g.apply(
    lambda grp: sliding_unique(grp, 'DE42_card_acceptor_id', pd.Timedelta(hours=24)))

print('Calculando unique_countries_last_24h ...')
df['unique_countries_last_24h'] = g.apply(
    lambda grp: sliding_unique(grp, 'DE19_acquirer_country_code', pd.Timedelta(hours=24)))

print('\nComercios/países únicos (24h) — fraude vs legítimo:')
for col in ['unique_merchants_last_24h', 'unique_countries_last_24h']:
    means = df.groupby('is_fraud')[col].mean().round(3)
    print(f'  {col:<30} legítimo={means[False]:.3f}  fraude={means[True]:.3f}')


Calculando unique_merchants_last_24h ...
Calculando unique_countries_last_24h ...



Comercios/países únicos (24h) — fraude vs legítimo:
  unique_merchants_last_24h      legítimo=0.139  fraude=1.622
  unique_countries_last_24h      legítimo=0.131  fraude=1.064


### 11.9 `is_night_transaction` — Transacción nocturna (00:00–05:00)

**Por qué importa:** el fraude de hospedaje frecuentemente ocurre de madrugada (check-in tardío, cargos post-evento). Es una señal débil pero constante.

In [19]:
df['is_night_transaction'] = df['hour_local'].between(0, 4).astype(int)

print('Transacciones nocturnas (0-4h) — fraude vs legítimo:')
print(df.groupby('is_fraud')['is_night_transaction'].mean().round(4))
print()
print('En hospedaje específicamente:')
print(df[df['is_hotel']].groupby('is_fraud')['is_night_transaction'].mean().round(4))


Transacciones nocturnas (0-4h) — fraude vs legítimo:
is_fraud
False    0.0839
True     0.1075
Name: is_night_transaction, dtype: float64

En hospedaje específicamente:
is_fraud
False    0.0824
True     0.1398
Name: is_night_transaction, dtype: float64


### 11.10 `amount_vs_max_ever_ratio` — Monto actual vs máximo histórico del cliente

**Por qué importa:** un cargo que supera el máximo histórico del cliente es altamente anómalo. Un ratio > 1 = **nunca antes el cliente gastó tanto**.

In [20]:
df['amount_vs_max_ever_ratio'] = (
    df['amount_usd'] / df['client_max_amount_before'].replace(0, np.nan)
)

print('Ratio monto actual / máximo histórico — fraude vs legítimo:')
print(df.groupby('is_fraud')['amount_vs_max_ever_ratio'].describe().round(3))
print()
# Proporción de txns donde se supera el máximo histórico
above_max = (df['amount_vs_max_ever_ratio'] > 1).groupby(df['is_fraud']).mean()
print('% txns que superan el máximo histórico del cliente:')
print(above_max.round(4))


Ratio monto actual / máximo histórico — fraude vs legítimo:
            count   mean      std  min    25%    50%    75%       max
is_fraud                                                             
False     91156.0  0.662    8.456  0.0  0.065  0.156  0.454  1600.586
True       4847.0  6.739  150.295  0.0  0.219  0.600  1.117  5813.300

% txns que superan el máximo histórico del cliente:
is_fraud
False    0.1074
True     0.2909
Name: amount_vs_max_ever_ratio, dtype: float64


## 12. Resumen Half 1 + guía para Half 2

### Half 1 completada 
| Variable | Correlación |is_fraud | Tipo |
|---|---|---|
| `time_since_last_txn_min` | Alta inversa | Velocidad |
| `txn_count_last_1h` | Alta | Velocidad |
| `txn_count_last_24h` | Alta | Velocidad |
| `txn_count_last_7d` | Media | Velocidad |
| `amount_zscore_customer` | Media | Desviación |
| `amount_zscore_channel` | Media | Desviación |
| `unique_merchants_last_24h` | Media | Diversidad |
| `unique_countries_last_24h` | Media | Diversidad |
| `is_night_transaction` | Baja-media | Temporal |
| `amount_vs_max_ever_ratio` | Media | Desviación |

### Half 2 — partir de `train_features.csv`

| # | Variable | Descripción |
|---|---|---|
| 11 | `hotel_amount_zscore` | Z-score del monto en hotel vs historial de hotel del cliente |
| 12 | `days_since_last_hotel_txn` | Días desde la última txn en hospedaje |
| 13 | `hotel_new_country` | Flag: hotel en país nunca visitado antes por el cliente |
| 14 | `merchant_txn_count_before` | Txns previas en el mismo comercio (`DE42_card_acceptor_id`) |
| 15 | `rapid_country_change` | País diferente al de la txn anterior, en < 24h |
| 16 | `amount_round_flag` | Monto múltiplo exacto de 50 o 100 (card-testing) |
| 17 | `client_weekend_rate_before` | % histórico de compras en fin de semana |
| 18 | `hotel_distance_zscore` | Z-score de distancia del hotel vs historial de hotel del cliente |
| 19 | `hour_deviation_from_usual` | Desviación de la hora actual vs hora media histórica del cliente |
| 20 | `client_mcc_diversity_before` | Nº de MCCs distintos visitados históricamente (entropía de gasto) |


In [21]:
half1_vars = [
    'time_since_last_txn_min', 'txn_count_last_1h', 'txn_count_last_24h',
    'txn_count_last_7d', 'amount_zscore_customer', 'amount_zscore_channel',
    'unique_merchants_last_24h', 'unique_countries_last_24h',
    'is_night_transaction', 'amount_vs_max_ever_ratio',
]

print('Correlación con is_fraud — Half 1 (valor absoluto):')
corr = df[half1_vars + ['is_fraud']].corr(numeric_only=True)['is_fraud'].drop('is_fraud')
print(corr.abs().sort_values(ascending=False).round(4).to_string())


Correlación con is_fraud — Half 1 (valor absoluto):
txn_count_last_1h            0.5285
txn_count_last_24h           0.5175
unique_merchants_last_24h    0.5174
unique_countries_last_24h    0.4412
txn_count_last_7d            0.2795
time_since_last_txn_min      0.1284
amount_vs_max_ever_ratio     0.0383
is_night_transaction         0.0183
amount_zscore_customer       0.0020
amount_zscore_channel        0.0017


## 13. Guardar dataset final con todas las features

## 14. Half 2 — Variables hotel-específicas y comportamiento avanzado

Variables orientadas específicamente a detectar patrones de fraude en hospedaje
y comportamientos atípicos del cliente.

### 14.1 `hotel_amount_zscore` — Z-score del monto en hotel vs historial hotelero

**Por qué importa:** un monto en hotel anómalo respecto a los *propios hoteles* del
cliente es más preciso que compararlo con su gasto general.

In [22]:
# Std histórica de monto en hotel por cliente (expanding + shift dentro del grupo)
gh = df.groupby(['client_id', 'is_hotel'])
df['client_hotel_std_before'] = gh['amount_usd'].transform(
    lambda x: x.expanding().std().shift(1)
)

df['hotel_amount_zscore'] = np.where(
    df['is_hotel'] & df['client_hotel_avg_amount_before'].notna(),
    (df['amount_usd'] - df['client_hotel_avg_amount_before'])
    / df['client_hotel_std_before'].replace(0, np.nan),
    np.nan
)

print('Z-score de monto en hotel — fraude vs legítimo (solo txns de hotel):')
hotel_mask = df['is_hotel']
print(df[hotel_mask].groupby('is_fraud')['hotel_amount_zscore'].describe().round(3))


Z-score de monto en hotel — fraude vs legítimo (solo txns de hotel):
           count   mean     std      min    25%    50%    75%      max
is_fraud                                                              
False     2651.0 -0.233  14.947 -456.088 -1.111 -0.119  0.918  387.457
True       163.0  0.875   9.449 -103.724 -0.564  0.877  2.453   24.873


### 14.2 `days_since_last_hotel_txn` — Días desde la última transacción en hotel

**Por qué importa:** un cliente que repite hotel muy rápido (< 1 día) puede estar
haciendo cargos fraudulentos consecutivos en el mismo establecimiento.

In [23]:
# Fecha/hora de la txn anterior en hotel (solo dentro del grupo is_hotel=True)
df['_prev_hotel_dt'] = gh['transaction_datetime'].shift(1)

df['days_since_last_hotel_txn'] = np.where(
    df['is_hotel'] & df['_prev_hotel_dt'].notna(),
    (df['transaction_datetime'] - df['_prev_hotel_dt']).dt.total_seconds() / 86400,
    -1  # -1 = primera txn en hotel o no es hotel
)
df.drop(columns=['_prev_hotel_dt'], inplace=True)

print('Días desde última txn en hotel — fraude vs legítimo (solo hotels con historial):')
mask_con_hist = df['is_hotel'] & (df['days_since_last_hotel_txn'] >= 0)
print(df[mask_con_hist].groupby('is_fraud')['days_since_last_hotel_txn'].describe().round(2))


Días desde última txn en hotel — fraude vs legítimo (solo hotels con historial):
           count   mean    std  min    25%    50%    75%     max
is_fraud                                                        
False     5194.0  41.35  34.87  0.0  13.45  31.96  60.40  176.68
True       270.0  32.43  36.59  0.0   1.89  22.09  48.88  173.80


### 14.3 `hotel_new_country` — Hotel en país nunca visitado antes por el cliente

**Por qué importa:** un hotel en un país completamente nuevo para el cliente
es una señal fuerte de fraude (especialmente tarjetas robadas usadas en el exterior).

In [24]:
# Para cada txn de hotel, verificar si DE19_acquirer_country_code
# ya apareció en txns de hotel ANTERIORES del mismo cliente
df['hotel_new_country'] = 0

hotel_df_idx = df[df['is_hotel']].index
hotel_sub = df.loc[hotel_df_idx, ['client_id','transaction_datetime',
                                    'DE19_acquirer_country_code']].copy()
hotel_sub = hotel_sub.sort_values(['client_id','transaction_datetime'])

seen = {}  # client_id -> set de países vistos
new_country_flags = []
for _, row in hotel_sub.iterrows():
    cid     = row['client_id']
    country = row['DE19_acquirer_country_code']
    if cid not in seen:
        seen[cid] = set()
    flag = 1 if country not in seen[cid] else 0
    new_country_flags.append(flag)
    seen[cid].add(country)

df.loc[hotel_df_idx, 'hotel_new_country'] = new_country_flags

print('Hotel en país nuevo para el cliente — fraude vs legítimo (solo hotels):')
print(df[df['is_hotel']].groupby('is_fraud')['hotel_new_country'].mean().round(4))


Hotel en país nuevo para el cliente — fraude vs legítimo (solo hotels):
is_fraud
False    0.5890
True     0.7229
Name: hotel_new_country, dtype: float64


### 14.4 `merchant_txn_count_before` — Txns previas en el mismo comercio

**Por qué importa:** `0` = comercio completamente nuevo para el cliente.
Fraudes en hotel frecuentemente ocurren en comercios que el cliente nunca visitó.

In [25]:
# cumcount por (client_id, DE42_card_acceptor_id) → 0-indexed
df['merchant_txn_count_before'] = df.groupby(
    ['client_id', 'DE42_card_acceptor_id']
).cumcount()

print('Txns previas en el mismo comercio — fraude vs legítimo:')
print(df.groupby('is_fraud')['merchant_txn_count_before'].describe().round(2))
print()
print('Comercio nuevo (0 txns previas) por is_fraud:')
print((df['merchant_txn_count_before'] == 0).groupby(df['is_fraud']).mean().round(4))


Txns previas en el mismo comercio — fraude vs legítimo:
            count  mean   std  min  25%  50%  75%  max
is_fraud                                              
False     94139.0  0.03  0.16  0.0  0.0  0.0  0.0  2.0
True       4861.0  0.03  0.17  0.0  0.0  0.0  0.0  2.0

Comercio nuevo (0 txns previas) por is_fraud:
is_fraud
False    0.9654
True     0.9620
Name: merchant_txn_count_before, dtype: float64


### 14.5 `rapid_country_change` — País diferente a la txn anterior en < 24h

**Por qué importa:** detecta *impossible travel* — un cliente no puede estar
físicamente en dos países distintos en pocas horas.

In [26]:
g = df.groupby('client_id')

# País de la transacción anterior
df['_prev_country'] = g['DE19_acquirer_country_code'].shift(1)

df['rapid_country_change'] = (
    (df['DE19_acquirer_country_code'] != df['_prev_country'])  # país diferente
    & (df['time_since_last_txn_min'] >= 0)                    # no es la primera
    & (df['time_since_last_txn_min'] < 1440)                  # en menos de 24h
).astype(int)
df.drop(columns=['_prev_country'], inplace=True)

print('Cambio rápido de país (<24h) — fraude vs legítimo:')
print(df.groupby('is_fraud')['rapid_country_change'].mean().round(4))
print()
print('En hospedaje específicamente:')
print(df[df['is_hotel']].groupby('is_fraud')['rapid_country_change'].mean().round(4))


Cambio rápido de país (<24h) — fraude vs legítimo:
is_fraud
False    0.0489
True     0.3755
Name: rapid_country_change, dtype: float64

En hospedaje específicamente:
is_fraud
False    0.0523
True     0.4819
Name: rapid_country_change, dtype: float64


### 14.6 `amount_round_flag` — Monto múltiplo de 50 o 100 USD

**Por qué importa:** en *card-testing* se usan montos redondos para verificar
que la tarjeta funciona antes de hacer cargos grandes.

In [27]:
df['amount_round_flag'] = (
    (df['amount_usd'] % 50 < 0.01) | (df['amount_usd'] % 50 > 49.99)
).astype(int)

print('Monto redondo (múltiplo de 50) — fraude vs legítimo:')
print(df.groupby('is_fraud')['amount_round_flag'].mean().round(4))


Monto redondo (múltiplo de 50) — fraude vs legítimo:
is_fraud
False    0.0005
True     0.0002
Name: amount_round_flag, dtype: float64


### 14.7 `client_weekend_rate_before` — Tasa histórica de compras en fin de semana

**Por qué importa:** si un cliente que nunca compra en fin de semana
de repente tiene una txn de hotel un sábado, es inusual.

In [28]:
weekend_int = df['day_of_week'].isin(['Sat', 'Sun']).astype(int)
cumsum_wk   = weekend_int.groupby(df['client_id']).cumsum()

df['client_weekend_rate_before'] = (
    (cumsum_wk - weekend_int)
    / df['client_txn_count_before'].replace(0, np.nan)
)

print('Tasa histórica de compras en fin de semana — fraude vs legítimo:')
print(df.groupby('is_fraud')['client_weekend_rate_before'].mean().round(4))


Tasa histórica de compras en fin de semana — fraude vs legítimo:
is_fraud
False    0.2762
True     0.2699
Name: client_weekend_rate_before, dtype: float64


### 14.8 `hotel_distance_zscore` — Z-score de distancia en hotel vs historial hotelero

**Por qué importa:** un hotel a 11,000 km cuando el cliente normalmente
se aloja a 50 km de su casa es una anomalía geográfica clara.

In [29]:
# Media y std de distancia histórica en hoteles (solo txns de hotel)
cumsum_dist  = gh['distance_from_home_km'].cumsum()
sum_dist_before = cumsum_dist - df['distance_from_home_km']

df['client_hotel_avg_dist_before'] = np.where(
    df['is_hotel'] & (df['client_hotel_txn_count_before'] > 0),
    sum_dist_before / df['client_hotel_txn_count_before'],
    np.nan
)
df['client_hotel_std_dist_before'] = gh['distance_from_home_km'].transform(
    lambda x: x.expanding().std().shift(1)
)

df['hotel_distance_zscore'] = np.where(
    df['is_hotel'] & df['client_hotel_avg_dist_before'].notna(),
    (df['distance_from_home_km'] - df['client_hotel_avg_dist_before'])
    / df['client_hotel_std_dist_before'].replace(0, np.nan),
    np.nan
)

print('Z-score de distancia en hotel — fraude vs legítimo (solo hotels con historial):')
print(df[df['is_hotel']].groupby('is_fraud')['hotel_distance_zscore'].describe().round(3))


Z-score de distancia en hotel — fraude vs legítimo (solo hotels con historial):
           count     mean       std      min    25%    50%    75%        max
is_fraud                                                                    
False     2589.0  161.659  1203.213 -185.969 -0.709 -0.553  1.209  38930.824
True       162.0  105.272   316.536  -15.352 -0.420  0.953  6.886   1962.002


### 14.9 `hour_deviation_from_usual` — Desviación de la hora vs hora habitual del cliente

**Por qué importa:** un cliente que siempre compra entre 9–18h y aparece
con un cargo de hotel a las 3am está fuera de su patrón habitual.

In [30]:
# Media histórica de la hora de transacción del cliente (antes de la actual)
cumsum_hour  = df['hour_local'].groupby(df['client_id']).cumsum()
df['client_avg_hour_before'] = (
    (cumsum_hour - df['hour_local'])
    / df['client_txn_count_before'].replace(0, np.nan)
)

df['hour_deviation_from_usual'] = (
    (df['hour_local'] - df['client_avg_hour_before']).abs()
)

print('Desviación de hora habitual — fraude vs legítimo:')
print(df.groupby('is_fraud')['hour_deviation_from_usual'].describe().round(3))
print()
print('En hospedaje:')
print(df[df['is_hotel']].groupby('is_fraud')['hour_deviation_from_usual'].mean().round(3))


Desviación de hora habitual — fraude vs legítimo:
            count   mean    std  min    25%    50%    75%   max
is_fraud                                                       
False     91156.0  5.186  3.631  0.0  2.182  4.600  7.643  23.0
True       4847.0  4.772  3.586  0.0  1.913  4.091  6.929  23.0

En hospedaje:
is_fraud
False    5.231
True     5.471
Name: hour_deviation_from_usual, dtype: float64


### 14.10 `client_mcc_diversity_before` — Nº de MCCs distintos visitados históricamente

**Por qué importa:** un cliente con historial diverso de comercios es
más predecible. Uno con 1 solo MCC en su historial que aparece en hotel es anómalo.

In [31]:
df['client_mcc_diversity_before'] = g['DE18_merchant_category_code'].transform(
    lambda x: x.expanding().apply(lambda s: s.iloc[:-1].nunique() if len(s) > 1 else 0,
                                   raw=False)
)

print('Diversidad de MCCs histórica — fraude vs legítimo:')
print(df.groupby('is_fraud')['client_mcc_diversity_before'].describe().round(2))


Diversidad de MCCs histórica — fraude vs legítimo:
            count  mean   std  min  25%  50%   75%   max
is_fraud                                                
False     95084.0  7.94  4.19  0.0  5.0  8.0  11.0  20.0
True       4919.0  8.55  3.96  0.0  6.0  9.0  12.0  20.0


## 15. Correlaciones Half 2 y resumen final

In [32]:
half2_vars = [
    'hotel_amount_zscore', 'days_since_last_hotel_txn', 'hotel_new_country',
    'merchant_txn_count_before', 'rapid_country_change', 'amount_round_flag',
    'client_weekend_rate_before', 'hotel_distance_zscore',
    'hour_deviation_from_usual', 'client_mcc_diversity_before',
]

valid = [c for c in half2_vars if c in df.columns]
print('Correlación con is_fraud — Half 2 (valor absoluto):')
corr = df[valid + ['is_fraud']].corr(numeric_only=True)['is_fraud'].drop('is_fraud')
print(corr.abs().sort_values(ascending=False).round(4).to_string())

all_fe = [
    'client_txn_count_before','client_avg_amount_before','client_std_amount_before',
    'client_max_amount_before','amount_vs_hist_avg','amount_over_hist_std',
    'client_intl_rate_before','client_channel_txn_count_before',
    'client_channel_avg_amount_before','amount_vs_channel_avg',
    'client_hotel_txn_count_before','client_hotel_avg_amount_before',
    'client_hotel_intl_rate_before','client_hotel_first_time',
    'hours_since_last_txn','is_rapid_succession','time_since_last_txn_min',
    'txn_count_last_1h','txn_count_last_24h','txn_count_last_7d',
    'amount_zscore_customer','amount_zscore_channel',
    'unique_merchants_last_24h','unique_countries_last_24h',
    'is_night_transaction','amount_vs_max_ever_ratio',
] + half2_vars
print(f'\nTotal features generadas (Half 1 + Half 2): {len(all_fe)}')
print(f'Total columnas en el dataset: {df.shape[1]}')


Correlación con is_fraud — Half 2 (valor absoluto):
rapid_country_change           0.2867
client_mcc_diversity_before    0.0319
hour_deviation_from_usual      0.0249
hotel_amount_zscore            0.0176
hotel_distance_zscore          0.0113
days_since_last_hotel_txn      0.0082
client_weekend_rate_before     0.0077
hotel_new_country              0.0071
amount_round_flag              0.0028
merchant_txn_count_before      0.0024

Total features generadas (Half 1 + Half 2): 36
Total columnas en el dataset: 112


## 16. Guardar dataset completo (Half 1 + Half 2)

In [33]:
from pathlib import Path
OUT = Path('../data/processed')

df.to_csv(OUT / 'features_dataset.csv', index=False)

train_fe = df[df['transaction_month'] < 6].copy()
test_fe  = df[df['transaction_month'] == 6].copy()
train_fe.to_csv(OUT / 'train_features.csv', index=False)
test_fe.to_csv( OUT / 'test_features_june.csv', index=False)

print('Guardado en data/processed/:')
print(f'  features_dataset.csv    → {df.shape[0]:,} filas × {df.shape[1]} cols')
print(f'  train_features.csv      → {len(train_fe):,} filas')
print(f'  test_features_june.csv  → {len(test_fe):,} filas')


Guardado en data/processed/:
  features_dataset.csv    → 100,003 filas × 112 cols
  train_features.csv      → 6,629 filas
  test_features_june.csv  → 936 filas
